# Module 3: AgentCore Memory (15 min)

Create an AgentCore Memory resource and wire both short-term and long-term memory into your agent. The agent will automatically extract facts and preferences from conversations and recall them in future sessions.

**Prerequisites:** Module 2 completed (agent connected to Gateway tools)

---

## How AgentCore Memory Works

AgentCore Memory provides two layers:
- **Short-term memory**: Conversation history within a session (like Workshop 1's session managers)
- **Long-term memory**: Extracted facts and preferences that persist across sessions

Long-term memory uses extraction strategies:
- `SEMANTIC` — Extracts factual statements ("Sarah's order was delayed")
- `USER_PREFERENCE` — Extracts preferences ("Customer prefers email communication")

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q --disable-pip-version-check strands-agents bedrock-agentcore boto3

In [ ]:
# Where this module fits in the harness you are building
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
os.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))
from workshop_utils import lego_progress, quiet_logs

quiet_logs()          # keep third-party SDK logging out of the teaching output
lego_progress(3)       # the harness tower so far - one brick per module

---

## Step 2: Create a Memory Resource

First, create an AgentCore Memory resource. This is a managed service that stores and retrieves memories for your agent.

### CLI alternative

The cell below uses boto3. With the `agentcore` starter toolkit you can create
the same memory resource from a terminal (these commands are real — run
`agentcore memory --help`):

```bash
# Create memory with semantic-fact + user-preference long-term strategies,
# waiting until it is ACTIVE
agentcore memory create customer_service_memory \
  --region us-east-1 \
  --event-expiry-days 90 \
  --strategies '[{"semanticMemoryStrategy":{"name":"semantic_facts","namespaces":["/users/{actorId}/facts"]}},{"userPreferenceMemoryStrategy":{"name":"user_preferences","namespaces":["/users/{actorId}/preferences"]}}]' \
  --wait

# Inspect / check status / list / delete
agentcore memory status <MEMORY_ID> --region us-east-1
agentcore memory list --region us-east-1
agentcore memory delete <MEMORY_ID> --region us-east-1
```


In [ ]:
import boto3
import json
import os

REGION = "us-east-1"

# Memory management (create/get/list/delete) is a control-plane operation, so
# it lives on the bedrock-agentcore-control client.
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)


def find_memory_id(name):
    """Return the id of the memory created with this name, paging through all results.

    list_memories returns only the generated id (name + "-<suffix>"), not the
    name, so we match on the "<name>-" prefix exactly to avoid matching a
    similarly named resource (e.g. customer_service_memory_v2)."""
    next_token = None
    while True:
        kwargs = {"nextToken": next_token} if next_token else {}
        page = control_client.list_memories(**kwargs)
        for m in page.get("memories", []):
            if m.get("id", "").startswith(name + "-"):
                return m["id"]
        next_token = page.get("nextToken")
        if not next_token:
            return None


# Create a memory resource with semantic-fact and user-preference extraction.
# Strategy names must match [a-zA-Z][a-zA-Z0-9_]{0,47} (letters, digits,
# underscores — no hyphens).
try:
    response = control_client.create_memory(
        name="customer_service_memory",
        description="Memory for customer service agent",
        eventExpiryDuration=90,  # days to retain raw conversation events (required)
        memoryStrategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "semantic_facts",
                    "namespaces": ["/users/{actorId}/facts"],
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "user_preferences",
                    "namespaces": ["/users/{actorId}/preferences"],
                }
            },
        ],
    )
    MEMORY_ID = response["memory"]["id"]
    print("✅ Memory resource created!")
    print(f"   Memory ID: {MEMORY_ID}")
    print(f"   Status: {response['memory']['status']}")
except (control_client.exceptions.ConflictException,
        control_client.exceptions.ValidationException) as e:
    # A duplicate memory name comes back as ConflictException OR as a
    # ValidationException ("Memory with name ... already exists") depending on
    # the API path — handle both as "reuse the existing one".
    if "already exists" not in str(e) and not isinstance(
        e, control_client.exceptions.ConflictException
    ):
        raise  # a real validation error (bad strategy/param), not a duplicate
    print("Memory already exists — looking it up...")
    MEMORY_ID = find_memory_id("customer_service_memory")
    if not MEMORY_ID:
        raise RuntimeError(
            "Memory reported as existing but was not found via list_memories. "
            "Check the AgentCore console."
        )
    print(f"   Found: {MEMORY_ID}")

# A new memory starts in CREATING and can't serve events/records until ACTIVE.
# Wait for it before the agent cells use it (CreateEvent/ListEvents/Retrieve).
import time
print("\nWaiting for the memory to be ACTIVE...")
memory_active = False
for _ in range(30):
    status = control_client.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if status == "ACTIVE":
        memory_active = True
        print("  Memory ACTIVE ✅")
        break
    if status == "FAILED":
        raise RuntimeError("Memory entered FAILED state — check the AgentCore console.")
    print(f"  status: {status} — waiting...")
    time.sleep(10)
if not memory_active:
    raise TimeoutError("Memory did not become ACTIVE within ~5 minutes.")

# Save this for Module 5 (and re-runs of this notebook):
print(f"\n👉 Module 5 needs this:  export BEDROCK_AGENTCORE_MEMORY_ID=\"{MEMORY_ID}\"")


---

## Step 3: Configure the Session Manager

The `AgentCoreMemorySessionManager` integrates directly with Strands. It handles both short-term conversation history and long-term memory retrieval/storage.

In [ ]:
from bedrock_agentcore.memory.integrations.strands.config import (
    AgentCoreMemoryConfig,
    RetrievalConfig,
)
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)

# Configure memory with retrieval namespaces
config = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id="session-001",
    actor_id="customer-sarah",
    retrieval_config={
        "/users/{actorId}/facts": RetrievalConfig(),
        "/users/{actorId}/preferences": RetrievalConfig(),
    },
)

session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=config,
    region_name=REGION,
)

print(f"✅ Session manager configured")
print(f"   Memory ID: {MEMORY_ID}")
print(f"   Session: session-001")
print(f"   Actor: customer-sarah")
print(f"   Namespaces: facts, preferences")

---

## Step 4: Create the Agent with Memory

Wire the session manager into the agent. The agent now automatically stores and retrieves memories.

In [ ]:
import sys
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from strands import Agent
from strands.tools.mcp import MCPClient

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
os.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))
from workshop_utils import ToolTraceHook, show_result  # live tracing + token metrics

# Gateway connection from Module 2
GATEWAY_ENDPOINT_URL = os.environ.get("AGENTCORE_GATEWAY_URL", "YOUR_GATEWAY_ENDPOINT_URL")

SYSTEM_PROMPT = (
    "You are a customer service agent for an e-commerce company. "
    "You help customers check their account info, order status, and process refunds. "
    "Be helpful, concise, and professional. "
    "Remember customer preferences and past interactions."
)

# Same proxy setup as Module 2: --region for SigV4 signing, env= forwards the
# parent environment (AWS_* plus HOME/AWS_PROFILE) so the proxy child can find
# credentials however they are configured.
proxy_env = os.environ.copy()

gateway_mcp = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx",
            args=["mcp-proxy-for-aws@latest", GATEWAY_ENDPOINT_URL, "--region", REGION],
            env=proxy_env,
        )
    )
)

# Open the MCP session and keep it open for the Session 1 and Session 2 turns
# below; the final cell closes it. Pass the resolved tools (not the client).
# Close any session left open by a previous run first, so re-running this cell
# never hits "client session is currently running".
try:
    gateway_mcp.__exit__(None, None, None)
except Exception:
    pass

gateway_mcp.__enter__()
gateway_tools = gateway_mcp.list_tools_sync()
agent = Agent(
    tools=gateway_tools,
    session_manager=session_manager,
    system_prompt=SYSTEM_PROMPT,
    hooks=[ToolTraceHook()],  # watch Gateway tool calls as memory turns run
)

print("✅ Agent created with AgentCore Memory (MCP session open)")


---

## Step 5: Multi-Turn Conversation (Session 1)

Have a conversation that establishes facts and preferences. Memory will extract and store them automatically.

In [ ]:
# Turn 1: Identify the customer
show_result(agent("Hi, I'm Sarah Johnson, customer C-1001."))


In [ ]:
# Turn 2: Express a preference (this is what long-term memory will extract)
show_result(agent("By the way, I prefer to be contacted by email rather than phone."))


In [ ]:
# Turn 3: Ask about orders
show_result(agent("Can you check my order status? I'm worried about the USB-C Hub."))


---

## Step 5b: Watch long-term memory get extracted

Short-term memory (the conversation above) is saved instantly. **Long-term memory is different:** AgentCore runs extraction *asynchronously* in the background, reading the raw conversation and distilling durable facts and preferences into the namespaces you configured.

Before we open a new session, let's wait for that extraction to finish and read the records straight from the Memory service — so you can see exactly what the agent will recall later. This is the "magic" of long-term memory made visible.

In [ ]:
# Wait for the async extraction, then read the long-term records directly.
# MemoryClient is the data-plane client the session manager uses under the hood;
# here we call it ourselves so we can SEE what was extracted.
from bedrock_agentcore.memory.client import MemoryClient

memory_client = MemoryClient(region_name=REGION)
ACTOR_ID = "customer-sarah"
PREF_NS = f"/users/{ACTOR_ID}/preferences"
FACTS_NS = f"/users/{ACTOR_ID}/facts"

print("Waiting for long-term extraction (this runs in the background)...")
# wait_for_memories polls the namespace until at least one record appears.
memory_client.wait_for_memories(
    memory_id=MEMORY_ID,
    namespace=PREF_NS,
    max_wait=180,
    poll_interval=15,
)


def show_records(label, namespace, query):
    records = memory_client.retrieve_memories(
        memory_id=MEMORY_ID, namespace=namespace, query=query, top_k=5
    )
    print(f"\n{label} ({len(records)} record(s)):")
    for r in records:
        content = r.get("content", r)
        text = content.get("text") if isinstance(content, dict) else content
        print(f"  • {text}")


show_records("🧠 Preferences extracted", PREF_NS, "how should we contact this customer")
show_records("🧠 Facts extracted", FACTS_NS, "customer details and history")
print("\n✅ These records live beyond the session. A brand-new session can recall them.")


---

## Step 6: New Session — Memory Persists

Create a new agent instance (simulating a restart) with a new session ID. The agent should recall facts and preferences from the previous session.

In [ ]:
# New session — simulates agent restart
config_session2 = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id="session-002",  # New session!
    actor_id="customer-sarah",  # Same customer
    retrieval_config={
        "/users/{actorId}/facts": RetrievalConfig(),
        "/users/{actorId}/preferences": RetrievalConfig(),
    },
)

session_manager_2 = AgentCoreMemorySessionManager(
    agentcore_memory_config=config_session2,
    region_name=REGION,
)

# Reuse the same (still-open) MCP session and its resolved tools.
agent_2 = Agent(
    tools=gateway_tools,
    session_manager=session_manager_2,
    system_prompt=(
        "You are a customer service agent for an e-commerce company. "
        "You help customers check their account info, order status, and process refunds. "
        "Be helpful, concise, and professional. "
        "Remember customer preferences and past interactions."
    ),
)

print("✅ New agent instance created (session-002)")
print("   This simulates a fresh restart — no conversation history.")
print("   But long-term memory should still have Sarah's preferences.")


In [ ]:
# The real test: ask something only MEMORY can answer — not the Gateway tools.
# There is no "preferences" tool, so if the agent knows Sarah's contact
# preference here, it can ONLY have come from long-term memory written in
# session-001. 💡 Watch for "email" in a brand-new session with no history.
show_result(
    agent_2(
        "Hi, it's Sarah Johnson (C-1001) again. Quick question: do you already "
        "have my preferred contact method on file, or should I tell you again?"
    ),
    label="Agent (new session)",
)


In [ ]:
# Close the MCP session opened in Step 4 (both agents reused it).
gateway_mcp.__exit__(None, None, None)
print("✅ MCP session closed")


---

## What's Next

Your agent now has persistent memory across sessions. In **Module 4**, you'll deploy the agent to AgentCore Runtime and read the traces, metrics, and logs it captures in CloudWatch — the runtime auto-instruments your agent, so you add one dependency and write no instrumentation code.